#**01: Install All the Required Packages**

In [1]:
!nvidia-smi

Thu Dec 25 07:20:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install -U langchain langchain-core langchain-community
%pip install -U transformers accelerate bitsandbytes huggingface_hub python-dotenv


#**04: Approach 1:  Access Models Hosted on Hugging Face Through API**

#**Text2Text Generation Models | Seq2Seq Models | Encoder-Decoder Models**

#**Apprach 02: Text Generation Models | Decoder Only Models**

#**05: Approach 02: Download Model Locally (Create Pipelines)**

#**Import All the Required Libraries**

In [ ]:
import os
import torch
from dotenv import load_dotenv

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline as hf_pipeline,
    BitsAndBytesConfig,
)

from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline

False

In [ ]:
HF_TOKEN = ""

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Create a .env file with HF_TOKEN=...")

print("HF_TOKEN loaded:", "YES" if HF_TOKEN else "NO")
print("CUDA available:", torch.cuda.is_available())


HF_TOKEN loaded: YES
CUDA available: True


In [6]:
model_id = "google/flan-t5-large"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
)

print("Model loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded.


In [8]:
gen_pipe = hf_pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)
print("Pipeline + LangChain wrapper ready.")


Device set to use cuda:0


Pipeline + LangChain wrapper ready.


/tmp/ipython-input-3304701736.py:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=gen_pipe)


In [9]:
prompt_company = PromptTemplate(
    input_variables=["product"],
    template=(
        "Suggest 8 premium, brandable company names for a business that makes {product}. "
        "Return only the names as a numbered list."
    ),
)

chain_company = prompt_company | llm

result = chain_company.invoke({"product": "colorful socks"})
print(result)


flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, flannel, f


In [12]:
prompt_player = PromptTemplate(
    input_variables=["name"],
    template=(
        "Write a concise 6-bullet bio of the famous footballer {name}. "
        "Keep it factual and high level."
    ),
)

chain_player = prompt_player | llm

result = chain_player.invoke({"name": "Virat Kohli"})
print(result)


Virat Kohli is an Indian footballer who plays as a midfielder for Indian Premier League side Mumbai Indians .


In [ ]:
# model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_id,
#     token=HF_TOKEN,
#     device_map="auto",
#     torch_dtype=torch.float16,
# )

# gen_pipe = hf_pipeline(
#     "text2text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=128,
# )

# llm = HuggingFacePipeline(pipeline=gen_pipe)
# print("Fallback FP16 model loaded.")